# 06 - Localisation, patch CNN and statistical baselines

**Owner:** Person 4


Three non-MIL views of the problem: a patch-level CNN probability map, the
attention-derived statistical features with ordinal logistic regression /
XGBoost baselines, and the localisation overlap scores that put a number on
"the heatmap looks right".


## Setup

In [ ]:
import os, sys, json
from pathlib import Path
os.environ.setdefault("MPLBACKEND", "Agg")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, torch
print("torch", torch.__version__)

## 1. Attention-vs-mask overlap (localisation score)

In [ ]:
from src.heatmaps import evaluate_localisation

# Uses any exported attention array + a mask image.
npz_dir = ROOT / "results/mock_heatmaps/arrays"
masks = ROOT / "data/mock/masks"
if npz_dir.exists() and masks.exists():
    import numpy as np
    example = sorted(npz_dir.glob("*.npz"))[0]
    payload = np.load(example, allow_pickle=True)
    mask_path = masks / f"{example.stem}.png"
    score = evaluate_localisation(payload["attention"], mask_path, payload["coords"], patch_size=256)
    print(example.stem, json.dumps(score, indent=2, default=float))
else:
    print("Run scripts/run_mock_demo.sh first to produce attention arrays.")

## 2. Statistical baselines over attention features

In [ ]:
from src.statistical_models import FEATURE_NAMES, OrdinalLogisticRegression, bag_statistics
print("x_stat features:", FEATURE_NAMES)

att_csv = next(iter(sorted((ROOT / "results").glob("mock_eval/attention_*.csv"))), None)
if att_csv:
    import subprocess, sys
    cmd = [sys.executable, "-m", "src.statistical_models",
           "--attention-csv", str(att_csv),
           "--manifest", str(ROOT / "data/mock/manifest.csv"),
           "--out-dir", str(ROOT / "results/mock_statistical")]
    res = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(res.stdout[-800:]); print(res.stderr[-500:])
else:
    print("No attention CSV yet.")

## 3. Patch CNN (benign vs malignant patch classifier)

In [ ]:
from src.models import PatchCNN, PatchCNNConfig

model = PatchCNN(PatchCNNConfig(width=16))
print("patch CNN parameters:", sum(p.numel() for p in model.parameters()))
print(model(torch.randn(2, 3, 256, 256)).shape)

print('''# training (needs a patch CSV: patch_path,label):
from src.models.patch_cnn import train_patch_cnn
train_patch_cnn("data/patch_labels/train.csv", "data/patch_labels/val.csv",
                out_path="results/metrics/patch_cnn.pt", epochs=5)''')

## Reporting notes

* Patch-level AUC/F1 requires patch labels that PANDA does not provide; the CNN
  is therefore a weakly supervised *proxy* and its numbers are reported as such.
* Overlap scores are coarse: PANDA has no pixel-level ISUP masks, so they use
  tissue/tumour masks and are a sanity check, not a segmentation metric.
* The statistical baseline answers "is MIL needed at all?" -- report it next to
  ABMIL / CLAM-SB / TransMIL.